In [1]:
from experiment import Experiment
from database import PatchDataset
from models import SnowCoverNetV1, LinearModel, DummyModel
from metrics import WeightedSoftDiceLoss, WeightedMSE, ComplexMetric, Accuracy, Precision, Recall, F1
from optimizer import AdamOptimizer
import pandas as pd

In [2]:
ds = PatchDataset(
    input_variables=[
        'inm_snow_cover', 'inm_swe', 'inm_t2m', 'inm_olr', 'inm_ww', 'inm_ts', 'inm_v850', 'inm_u850', 'inm_tp', 'inm_h500', 'inm_hlt', 'inm_mslp',
        'inm_cos_lat', 'inm_sin_lon', 'inm_cos_lon', 'inm_lead_time', 'inm_cos_day', 'inm_sin_day', 'inm_year_norm', 'inm_day', 'inm_lat', 'inm_lon',
        'snow_cover', 'sd', 't2m', 'tp', 'pt', 'sst', 'lsm', 'glaicer', 'z', 'sdor',
        'cos_lat', 'sin_lon', 'cos_lon', 'cos_day', 'sin_day', 'year_norm'
    ],
    target_variables=['snow_cover', 'lat', 'lon', 'day', 'year', 'inm_lead_time'],
    modes={
        'train': {"t_min": '19910101', 't_max': '20201231', 'epoch_size': 500, 'batch_size': 8},
        'test': {"t_min": '20240801', 't_max': '20260430', 'epoch_size': 100, 'batch_size': 8},
    },
    era_scales=[{'id': 'local', 'xSize': 64, 'ySize': 64, 'tSize': 7, 'xyStep': 1, 'tStep': 1},
                {'id': 'regional', 'xSize': 16, 'ySize': 16, 'tSize': 8, 'xyStep': 4, 'tStep': 7},
                {'id': 'global', 'xSize': 88, 'ySize': 16, 'tSize': 6, 'xyStep': 16, 'tStep': 30, 'fixY': 4}],
    inm_scales=[{'id': 'regional', 'xSize': 16, 'ySize': 16, 'tSize': 28, 'xyStep': 1, 'tStep': 1},
                {'id': 'global', 'xSize': 16, 'ySize': 16, 'tSize': 4, 'xyStep': 4, 'tStep': 7, 'fixY': 4}],
    target_scale={'xSize': 64, 'ySize': 64, 'tSize': 28, 'xyStep': 1, 'tStep': 1},
    mask='snow',
    num_workers=8
)

In [2]:
ds = PatchDataset(
    input_variables=[
        'inm_snow_cover',
        'snow_cover',
        'cos_lat', 'sin_lon', 'cos_lon', 'cos_day', 'sin_day', 'year_norm'
    ],
    target_variables=['snow_cover', 'lat', 'lon', 'day', 'year'],
    modes={
        'train': {"t_min": '19910101', 't_max': '20201231', 'epoch_size': 500, 'batch_size': 8},
        'test': {"t_min": '20240801', 't_max': '20260430', 'epoch_size': 500, 'batch_size': 8},
    },
    era_scales=[{'id': 'local', 'xSize': 64, 'ySize': 64, 'tSize': 7, 'xyStep': 1, 'tStep': 1},
                {'id': 'regional', 'xSize': 16, 'ySize': 16, 'tSize': 8, 'xyStep': 4, 'tStep': 7},
                {'id': 'global', 'xSize': 88, 'ySize': 16, 'tSize': 6, 'xyStep': 16, 'tStep': 30, 'fixY': 4}],
    inm_scales=[{'id': 'regional', 'xSize': 16, 'ySize': 16, 'tSize': 28, 'xyStep': 1, 'tStep': 1},
                {'id': 'global', 'xSize': 16, 'ySize': 16, 'tSize': 4, 'xyStep': 4, 'tStep': 7, 'fixY': 4}],
    target_scale={'xSize': 64, 'ySize': 64, 'tSize': 28, 'xyStep': 1, 'tStep': 1},
    mask='snow',
    num_workers=8
)

In [ ]:
model = DummyModel('inm_snow_cover_regional', size=(64, 64))
metric = ComplexMetric(metrics={'dice': WeightedSoftDiceLoss('snow_cover'),
                                'accuracy': Accuracy('snow_cover'),
                                'precision': Precision('snow_cover'),
                                'recall': Recall('snow_cover'),
                                'f1': F1('snow_cover')}, loss_metric='dice')
optimizer = AdamOptimizer(lr=0.001)
experiment = Experiment(f'test', ds, model, metric, optimizer, modes=['test'])
experiment.run(1)

In [3]:
model = DummyModel('inm_snow_cover_regional', size=(64, 64))
metric = ComplexMetric(metrics={'dice': WeightedSoftDiceLoss('snow_cover'),
                                'accuracy': Accuracy('snow_cover'),
                                'precision': Precision('snow_cover'),
                                'recall': Recall('snow_cover'),
                                'f1': F1('snow_cover')}, loss_metric='dice')
optimizer = AdamOptimizer(lr=0.001)
experiment = Experiment(f'snow_cover_base', ds, model, metric, optimizer, modes=['test'])
experiment.run(1)

test  epoch   1:  30%|###       | 30/100 [00:00<?, ?batch/s]

In [6]:
def count_parameters(model):
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    non_trainable_params = total_params - trainable_params
    
    print(f"Всего параметров: {total_params:,}")
    print(f"Обучаемых: {trainable_params:,}")
    print(f"Необучаемых: {non_trainable_params:,}")
    
    return total_params, trainable_params, non_trainable_params

spatial = ['cos_lat', 'cos_lon', 'sin_lon']
temporal = ['cos_day', 'sin_day', 'year_norm']
num_stats = 4
channels = 10 + len(spatial)
inm_channels = num_stats*11 + 1 + len(spatial)
inm_temporal=temporal + ['lead_time']
model = SnowCoverNetV1(
    variable_encoders={var: f'../experiments/ensemble_encoder_{var}'
                       for var in ['t2m', 'swe', 'olr', 'hlt', 'mslp', 'ts', 'tp', 'u850', 'v850', 'ww', 'h500']},
    scales={
        'local':        {'channels': channels,     'blocks': 4, 't_blocks': 0, 'spatial': spatial, 'temporal': temporal, 'skip': True},
        'regional':     {'channels': channels,     'blocks': 3, 't_blocks': 0, 'spatial': spatial, 'temporal': temporal},
        'global':       {'channels': channels,     'blocks': 3, 't_blocks': 0, 'spatial': spatial, 'temporal': temporal},
        'inm_regional': {'channels': inm_channels, 'blocks': 3, 't_blocks': 2, 'spatial': spatial, 'temporal': inm_temporal, 'target': True, 'skip': True},
        'inm_global':   {'channels': inm_channels, 'blocks': 3, 't_blocks': 0, 'spatial': spatial, 'temporal': inm_temporal},
    },
    token_size=256,
)
count_parameters(model)
metric = ComplexMetric(metrics={'dice': WeightedSoftDiceLoss('snow_cover'),
                                'accuracy': Accuracy('snow_cover'),
                                'precision': Precision('snow_cover'),
                                'recall': Recall('snow_cover'),
                                'f1': F1('snow_cover')}, loss_metric='dice')
optimizer = AdamOptimizer(lr=0.001)
experiment = Experiment(f'snow_cover_v1', ds, model, metric, optimizer, modes=['train', 'test'])
experiment.run(25, save_every_n_epochs=1)

Всего параметров: 14,821,566
Обучаемых: 14,605,207
Необучаемых: 216,359


train epoch   1:   0%|          | 0/500 [00:00<?, ?batch/s]

test  epoch   1:   0%|          | 0/100 [00:00<?, ?batch/s]

train epoch   2:   0%|          | 0/500 [00:00<?, ?batch/s]

test  epoch   2:   0%|          | 0/100 [00:00<?, ?batch/s]

...

test  epoch  23:   0%|          | 0/100 [00:00<?, ?batch/s]

train epoch  24:   0%|          | 0/500 [00:00<?, ?batch/s]

test  epoch  24:   0%|          | 0/100 [00:00<?, ?batch/s]

train epoch  25:   0%|          | 0/500 [00:00<?, ?batch/s]

test  epoch  25:   0%|          | 0/100 [00:00<?, ?batch/s]